In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
from pathlib import Path
import numpy as np  # linear algebra
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/kkhandekar/environmental-vs-ai-startups-india-eda/src/small_bench/checkpoints/pre_cell_0.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 0 ###

benchmark_name = "environmental-vs-ai-startups-india-eda"
# load & cleanup
file = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "indian-startup-recognized-by-dpiit" / "Startup_Counts_Across_India.csv"
df = pd.read_csv(file)
factor = 30
df = pd.concat([df] * factor)
df.info()

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/kkhandekar/environmental-vs-ai-startups-india-eda/src/small_bench/checkpoints/pre_cell_1.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 1 ###

df.drop("S No.", axis=1, inplace=True)
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)

# view
df.head()

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/kkhandekar/environmental-vs-ai-startups-india-eda/src/small_bench/checkpoints/pre_cell_2.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 2 ###

# Industry sub-categories for environmental & AI startups
env = ["Agriculture", "Green Technology", "Renewable Energy", "Waste Management"]
ai = ["AI", "Robotics", "Computer Vision"]

# combined df - environmental & AI startups only
df_ea = df.loc[(df["Industry"].isin(env)) | (df["Industry"].isin(ai))].reset_index(
    drop=True, inplace=False
)


# custom function to set Main Industry
def set_MainIndustry(ind):
    if ind in env:
        return "ENV"
    else:
        return "AI"


# adding a new column
df_ea["MainIndustry"] = df_ea.Industry.apply(lambda x: set_MainIndustry(x))

# basic stats
print(
    f"A total of {df_ea.shape[0]} startups were started in India between 2016 & 2022, out of which {df_ea.groupby('MainIndustry').size()['ENV']} are environmental related & {df_ea.groupby('MainIndustry').size()['AI']} are AI startups."
)